# Exploratory Data Analysis — E-Commerce Customer Analytics
**Dataset:** Customer Analytics (Kaggle)  
**Mata Kuliah:** Visualisasi Data  
**Semester:** Genap 2025/2026  

---


## 1. Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

df_raw = pd.read_csv('Train.csv')
print("Dataset shape:", df_raw.shape)
df_raw.head()


: 

## 2. Informasi Dataset

In [ ]:
print("=" * 50)
print("INFO DATASET")
print("=" * 50)
df_raw.info()


In [ ]:
print("=" * 50)
print("STATISTIK DESKRIPTIF")
print("=" * 50)
df_raw.describe().round(2)


## 3. Cek Missing Value & Duplikat

In [ ]:
print("Missing Values per Kolom:")
print(df_raw.isnull().sum())
print()
print(f"Jumlah baris duplikat: {df_raw.duplicated().sum()}")


## 4. Data Preprocessing
Langkah preprocessing yang dilakukan:
- Hapus missing value (jika ada)
- Hapus duplikat (jika ada)
- Encode kolom target `Reached.on.Time_Y.N` → `Delivery_Status`
- Feature engineering: `Weight_Category` dan `Discount_Category`


In [ ]:
df = df_raw.copy()

# Hapus missing & duplikat
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# Encode target
df['Delivery_Status'] = df['Reached.on.Time_Y.N'].map({1: 'On Time', 0: 'Late'})
df.drop(columns=['Reached.on.Time_Y.N'], inplace=True)

# Feature engineering: Weight Category
bins_w = [0, 2000, 4000, 6000, 8000]
labels_w = ['Very Light (<2kg)', 'Light (2-4kg)', 'Medium (4-6kg)', 'Heavy (>6kg)']
df['Weight_Category'] = pd.cut(df['Weight_in_gms'], bins=bins_w, labels=labels_w)

# Feature engineering: Discount Category
bins_d = [-1, 10, 30, 60, 100]
labels_d = ['Low (0-10%)', 'Medium (11-30%)', 'High (31-60%)', 'Very High (>60%)']
df['Discount_Category'] = pd.cut(df['Discount_offered'], bins=bins_d, labels=labels_d)

print("Shape setelah preprocessing:", df.shape)
print("Kolom:", df.columns.tolist())
df.head()


## 5. Distribusi Variabel Kategorik

In [ ]:
cat_cols = ['Warehouse_block', 'Mode_of_Shipment', 'Product_importance',
            'Gender', 'Delivery_Status']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=axes[i], palette='Set2')
    axes[i].set_title(f'Distribusi: {col}')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Jumlah')
    for bar in axes[i].patches:
        axes[i].annotate(f'{int(bar.get_height())}',
                         (bar.get_x() + bar.get_width()/2, bar.get_height()),
                         ha='center', va='bottom', fontsize=9)

axes[5].set_visible(False)
plt.suptitle('Distribusi Variabel Kategorik', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_categorical_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInsight: Mayoritas pengiriman menggunakan kapal (Ship). Warehouse F paling banyak. ~60% pengiriman On Time.")


## 6. Distribusi Variabel Numerik

In [ ]:
num_cols = ['Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product',
            'Prior_purchases', 'Discount_offered', 'Weight_in_gms']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.1f}')
    axes[i].set_title(f'Distribusi: {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')
    axes[i].legend(fontsize=9)

plt.suptitle('Distribusi Variabel Numerik', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_numeric_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInsight: Weight_in_gms berdistribusi hampir seragam. Discount_offered condong ke kiri (banyak diskon rendah). Customer_care_calls rata-rata 4 panggilan.")


## 7. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap — Variabel Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInsight: Tidak ada korelasi sangat kuat antar variabel numerik. Discount_offered dan Cost_of_the_Product memiliki korelasi negatif lemah.")


## 8. Delivery Status vs Variabel Numerik (Boxplot)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='Delivery_Status', y=col, palette={'On Time':'#2ecc71','Late':'#e74c3c'}, ax=axes[i])
    axes[i].set_title(f'{col} vs Delivery Status')
    axes[i].set_xlabel('Delivery Status')
    axes[i].set_ylabel(col)

plt.suptitle('Perbandingan Numerik berdasarkan Delivery Status', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_boxplot_delivery.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInsight: Produk yang terlambat cenderung memiliki Discount_offered lebih tinggi dan Weight_in_gms lebih berat.")


## 9. Analisis Warehouse Block vs Delivery Status

In [ ]:
ct = pd.crosstab(df['Warehouse_block'], df['Delivery_Status'], normalize='index') * 100

ax = ct.plot(kind='bar', figsize=(10, 5), color=['#e74c3c', '#2ecc71'], edgecolor='white')
ax.set_title('Persentase Delivery Status per Warehouse Block', fontsize=14, fontweight='bold')
ax.set_xlabel('Warehouse Block')
ax.set_ylabel('Persentase (%)')
ax.legend(title='Status')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('eda_warehouse_delivery.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInsight: Semua warehouse memiliki pola delivery yang relatif mirip (~60% On Time), tidak ada warehouse yang signifikan lebih buruk.")


## 10. Discount Category vs Delivery Status

In [ ]:
ct2 = pd.crosstab(df['Discount_Category'], df['Delivery_Status'], normalize='index') * 100

ax = ct2.plot(kind='bar', figsize=(10, 5), color=['#e74c3c', '#2ecc71'], edgecolor='white')
ax.set_title('Delivery Status berdasarkan Kategori Diskon', fontsize=14, fontweight='bold')
ax.set_xlabel('Kategori Diskon')
ax.set_ylabel('Persentase (%)')
ax.legend(title='Status')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('eda_discount_delivery.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInsight: Produk dengan diskon Very High (>60%) justru paling banyak terlambat. Kemungkinan produk murah/berdiskon tinggi diprioritaskan lebih rendah dalam pengiriman.")


## 11. Weight Category vs Delivery Status

In [ ]:
ct3 = pd.crosstab(df['Weight_Category'], df['Delivery_Status'], normalize='index') * 100

ax = ct3.plot(kind='bar', figsize=(10, 5), color=['#e74c3c', '#2ecc71'], edgecolor='white')
ax.set_title('Delivery Status berdasarkan Berat Produk', fontsize=14, fontweight='bold')
ax.set_xlabel('Kategori Berat')
ax.set_ylabel('Persentase (%)')
ax.legend(title='Status')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('eda_weight_delivery.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInsight: Produk Very Light justru paling tinggi persentase terlambatnya. Produk berat (Heavy) lebih sering On Time.")


## 12. Customer Care Calls vs Delivery Status

In [ ]:
avg_calls = df.groupby('Delivery_Status')['Customer_care_calls'].mean().reset_index()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(avg_calls['Delivery_Status'], avg_calls['Customer_care_calls'],
              color=['#e74c3c', '#2ecc71'], edgecolor='white', width=0.5)
ax.set_title('Rata-rata Customer Care Calls vs Delivery Status', fontsize=13, fontweight='bold')
ax.set_ylabel('Rata-rata Panggilan')
ax.set_xlabel('Delivery Status')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.2f}', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('eda_calls_delivery.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInsight: Pengiriman yang Late rata-rata menerima lebih banyak customer care calls — menunjukkan ketidakpuasan pelanggan.")


## 13. Simpan Dataset Hasil Preprocessing

In [ ]:
df.to_csv('Train_preprocessed.csv', index=False)
print("Dataset preprocessed berhasil disimpan: Train_preprocessed.csv")
print("Shape final:", df.shape)
print("Kolom:", df.columns.tolist())


## 14. Summary EDA

### Temuan Utama:
1. **Dataset bersih** — tidak ada missing value maupun duplikat.
2. **~60% pengiriman On Time**, ~40% Late.
3. **Mode pengiriman Ship** mendominasi (>70% dari total).
4. **Diskon tinggi (>60%)** berkorelasi dengan keterlambatan pengiriman.
5. **Produk sangat ringan (<2kg)** justru memiliki persentase keterlambatan tertinggi.
6. **Pelanggan yang mengalami keterlambatan** melakukan lebih banyak customer care calls.
7. **Tidak ada korelasi kuat** antar variabel numerik — setiap fitur bersifat independen.

### Kolom yang Ditambahkan (Feature Engineering):
- `Delivery_Status`: menggantikan `Reached.on.Time_Y.N` dengan label 'On Time' / 'Late'
- `Weight_Category`: kategori berat produk (4 level)
- `Discount_Category`: kategori tingkat diskon (4 level)
